In [ ]:
import os
import glob
import re
import xarray as xr
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

class KeralaRainfallPreprocessor:
    def __init__(self, shp_path):
        # Load shapefile
        india_states = gpd.read_file(shp_path)

        # Select Kerala
        self.kerala_shape = india_states[
            (india_states["admin"] == "India") &
            (india_states["name"] == "Kerala")
        ].to_crs(epsg=4326)

    def process_netcdf(self, base_dir):
        file_paths = sorted(glob.glob(os.path.join(base_dir, "*.nc")))
        print(f"Found {len(file_paths)} NetCDF files.")

        jjas_all = []
        ond_all  = []

        for file_path in file_paths:
            try:
                year_match = re.search(r'(\d{4})', os.path.basename(file_path))
                if not year_match:
                    continue
                year = int(year_match.group(1))

                print(f"Processing {year}...", end=' ')

                ds = xr.open_dataset(file_path)

                # ---- Extract seasons ----
                jjas = ds.sel(TIME=ds['TIME.month'].isin([6,7,8,9]))
                ond  = ds.sel(TIME=ds['TIME.month'].isin([10,11,12]))

                # ---- Aggregate rainfall ----
                jjas_sum = jjas['RAINFALL'].sum(dim='TIME')
                ond_sum  = ond['RAINFALL'].sum(dim='TIME')

                # ---- Convert to DataFrame ----
                df_jjas = jjas_sum.to_dataframe().reset_index()
                df_ond  = ond_sum.to_dataframe().reset_index()

                df_jjas['YEAR'] = year
                df_ond['YEAR']  = year

                df_jjas.rename(columns={'RAINFALL': 'JJAS_SUM'}, inplace=True)
                df_ond.rename(columns={'RAINFALL': 'OND_SUM'}, inplace=True)

                # ---- Convert to GeoDataFrame ----
                def clip_to_kerala(df, value_col):
                    geometry = [
                        Point(lon, lat)
                        for lon, lat in zip(df["LONGITUDE"], df["LATITUDE"])
                    ]

                    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

                    gdf_kerala = gpd.sjoin(
                        gdf,
                        self.kerala_shape,
                        predicate="intersects",
                        how="inner"
                    )

                    # ✅ Clean and explicit
                    return gdf_kerala[['LATITUDE', 'LONGITUDE', 'YEAR', value_col]]

                df_jjas = clip_to_kerala(df_jjas, 'JJAS_SUM')
                df_ond  = clip_to_kerala(df_ond, 'OND_SUM')

                jjas_all.append(df_jjas)
                ond_all.append(df_ond)

                print(f"JJAS:{len(df_jjas)} OND:{len(df_ond)}")

                ds.close()

            except Exception as e:
                print(f"Error processing {file_path}: {e}")

        # ---- Combine ----
        jjas_df = pd.concat(jjas_all, ignore_index=True)
        ond_df  = pd.concat(ond_all, ignore_index=True)

        # ---- Sort ----
        jjas_df = jjas_df.sort_values(by=['LATITUDE', 'LONGITUDE', 'YEAR']).reset_index(drop=True)
        ond_df  = ond_df.sort_values(by=['LATITUDE', 'LONGITUDE', 'YEAR']).reset_index(drop=True)

        return jjas_df, ond_df

    def save(self, jjas_df, ond_df, save_dir="."):
        jjas_path = os.path.join(save_dir, "kerala_jjas.csv")
        ond_path  = os.path.join(save_dir, "kerala_ond.csv")

        jjas_df.to_csv(jjas_path, index=False)
        ond_df.to_csv(ond_path, index=False)

        print("\nSaved:")
        print(jjas_path)
        print(ond_path)


In [ ]:

base_dir =  r"rain data 1901-2022" # Update this path to your NetCDF files directory
shp_path = r"./ne_10m_admin_1_states_provinces\ne_10m_admin_1_states_provinces.shp"

processor = KeralaRainfallPreprocessor(shp_path)
jjas_df, ond_df = processor.process_netcdf(base_dir)

processor.save(jjas_df, ond_df)

Found 123 NetCDF files.
Processing 1901... JJAS:46 OND:46
Processing 1902... JJAS:46 OND:46
Processing 1903... JJAS:46 OND:46
Processing 1904... JJAS:46 OND:46
Processing 1905... JJAS:46 OND:46
Processing 1906... JJAS:46 OND:46
Processing 1907... JJAS:46 OND:46
Processing 1908... JJAS:46 OND:46
Processing 1909... JJAS:46 OND:46
Processing 1910... JJAS:46 OND:46
Processing 1911... JJAS:46 OND:46
Processing 1912... JJAS:46 OND:46
Processing 1913... JJAS:46 OND:46
Processing 1914... JJAS:46 OND:46
Processing 1915... JJAS:46 OND:46
Processing 1916... JJAS:46 OND:46
Processing 1917... JJAS:46 OND:46
Processing 1918... JJAS:46 OND:46
Processing 1919... JJAS:46 OND:46
Processing 1920... JJAS:46 OND:46
Processing 1921... JJAS:46 OND:46
Processing 1922... JJAS:46 OND:46
Processing 1923... JJAS:46 OND:46
Processing 1924... JJAS:46 OND:46
Processing 1925... JJAS:46 OND:46
Processing 1926... JJAS:46 OND:46
Processing 1927... JJAS:46 OND:46
Processing 1928... JJAS:46 OND:46
Processing 1929... JJAS: